## Load WQP indexing to production

In [ ]:
import os,sys,psycopg2,tempfile,git;
import requests,csv,codecs,datetime;
from contextlib import closing;

sys.path.append(os.path.join(os.path.expanduser('~'),'notebooks'));
import common;

dbse = os.environ['POSTGRESQL_DB'];
host = os.environ['POSTGRESQL_HOST'];
port = os.environ['POSTGRESQL_PORT'];
user = 'cipsrv';
pswd = os.environ['POSTGRESQL_CIP_PASS'];

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

cs = "dbname=%s user=%s password=%s host=%s port=%s" % (
     dbse
    ,user
    ,pswd
    ,host
    ,port
);

try:
    conn = psycopg2.connect(cs);
except:
    raise Exception("database connection error");

print("database is ready");
print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

with closing(conn.cursor()) as cursor:
    cursor.execute("""
        SELECT
         'wqp_attr' AS tbl
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_attr) AS orig
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_attr) AS repl
        UNION ALL SELECT
         'wqp_cip'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_cip)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_cip)
        UNION ALL SELECT
         'wqp_cip_geo'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_cip_geo)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_cip_geo)
        UNION ALL SELECT
         'wqp_control'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_control)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_control)
        UNION ALL SELECT
         'wqp_huc12'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_huc12)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_huc12)
        UNION ALL SELECT
         'wqp_huc12_geo'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_huc12_geo)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_huc12_geo)
        UNION ALL SELECT
         'wqp_rad_evt2meta'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_rad_evt2meta)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_rad_evt2meta)
        UNION ALL SELECT
         'wqp_rad_p'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_rad_p)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_rad_p)
        UNION ALL SELECT
         'wqp_sfid'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_sfid)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_sfid)
        UNION ALL SELECT
         'wqp_srccip'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_src2cip)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_src2cip)
        UNION ALL SELECT
         'wqp_src_p'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_src_p)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_src_p)
        
    """);

    for row in cursor:
        print(f"{row[0]} orig:{row[1]} repl:{row[2]}")

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

with closing(conn.cursor()) as cursor:
    cursor.execute("TRUNCATE TABLE cipsrv_owld.wqp_attr");
    cursor.execute("TRUNCATE TABLE cipsrv_owld.wqp_cip");
    cursor.execute("TRUNCATE TABLE cipsrv_owld.wqp_cip_geo");
    cursor.execute("TRUNCATE TABLE cipsrv_owld.wqp_control");
    cursor.execute("TRUNCATE TABLE cipsrv_owld.wqp_huc12");
    cursor.execute("TRUNCATE TABLE cipsrv_owld.wqp_huc12_geo");
    cursor.execute("TRUNCATE TABLE cipsrv_owld.wqp_rad_a");
    cursor.execute("TRUNCATE TABLE cipsrv_owld.wqp_rad_evt2meta");
    cursor.execute("TRUNCATE TABLE cipsrv_owld.wqp_rad_l");
    cursor.execute("TRUNCATE TABLE cipsrv_owld.wqp_rad_metadata");
    cursor.execute("TRUNCATE TABLE cipsrv_owld.wqp_rad_p");
    cursor.execute("TRUNCATE TABLE cipsrv_owld.wqp_rad_srccit");
    cursor.execute("TRUNCATE TABLE cipsrv_owld.wqp_sfid");
    cursor.execute("TRUNCATE TABLE cipsrv_owld.wqp_src2cip");
    cursor.execute("TRUNCATE TABLE cipsrv_owld.wqp_src_a");
    cursor.execute("TRUNCATE TABLE cipsrv_owld.wqp_src_l");
    cursor.execute("TRUNCATE TABLE cipsrv_owld.wqp_src_p");

    conn.commit();

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

with closing(conn.cursor()) as cursor:
    cursor.execute("INSERT INTO cipsrv_owld.wqp_attr         SELECT * FROM cipdev_owld.wqp_attr");
    cursor.execute("INSERT INTO cipsrv_owld.wqp_cip          SELECT * FROM cipdev_owld.wqp_cip");
    cursor.execute("INSERT INTO cipsrv_owld.wqp_cip_geo      SELECT * FROM cipdev_owld.wqp_cip_geo");
    cursor.execute("INSERT INTO cipsrv_owld.wqp_control      SELECT * FROM cipdev_owld.wqp_control");
    cursor.execute("INSERT INTO cipsrv_owld.wqp_huc12        SELECT * FROM cipdev_owld.wqp_huc12");
    cursor.execute("INSERT INTO cipsrv_owld.wqp_huc12_geo    SELECT * FROM cipdev_owld.wqp_huc12_geo");
    cursor.execute("INSERT INTO cipsrv_owld.wqp_rad_evt2meta SELECT * FROM cipdev_owld.wqp_rad_evt2meta");
    cursor.execute("INSERT INTO cipsrv_owld.wqp_rad_metadata SELECT * FROM cipdev_owld.wqp_rad_metadata");
    cursor.execute("INSERT INTO cipsrv_owld.wqp_rad_p        SELECT * FROM cipdev_owld.wqp_rad_p");
    cursor.execute("INSERT INTO cipsrv_owld.wqp_sfid         SELECT * FROM cipdev_owld.wqp_sfid");
    cursor.execute("INSERT INTO cipsrv_owld.wqp_src2cip      SELECT * FROM cipdev_owld.wqp_src2cip");
    cursor.execute("INSERT INTO cipsrv_owld.wqp_src_p        SELECT * FROM cipdev_owld.wqp_src_p");

    conn.commit();

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");